# Lezione 10 — Portare modelli in produzione con TensorFlow Extended (TFX)

## Obiettivi di questa lezione

- Comprendere gli step di una pipeline di machine learning in produzione
- Conoscere i componenti di TensorFlow Extended (TFX)
- Creare una pipeline semplice con TFX per automatizzare l'addestramento e l'evaluazione di modelli
- Comprendere il ruolo del versionamento dei dati e dei modelli

## Da notebook a produzione

Finora addestravi un modello chiamando `fit` e basta. Ma in un'azienda reale, serve molto di più:

- **Automatizzazione**: quando arrivano nuovi dati, il modello si allena automaticamente
- **Versionamento**: sia il codice che i dati e i modelli devono avere versioni tracciabili
- **Validazione**: non tutti i modelli addestrati vanno in produzione (potrebbero essere peggiori del precedente)
- **Monitoraggio**: una volta distribuito, il modello va osservato nel tempo (il comportamento dei dati potrebbe cambiare)

## Che cos'è un pipeline di TFX

Un **pipeline** di TFX è una sequenza di **componenti** orchestrati. I principali sono:

- **ExampleGen**: carica i dati da una fonte (database, file, API)
- **StatisticsGen**: calcola le statistiche dei dati (media, distribuzione, ecc.)
- **SchemaGen**: deduce lo schema (quali colonne, quali tipi)
- **ExampleValidator**: controlla che i dati corrispondano allo schema
- **Transform**: preprocessing (normalizzazione, encoding, ecc.)
- **Trainer**: allena il modello
- **Evaluator**: valuta il modello su un set di test
- **ModelValidator**: controlla se il modello è abbastanza buono per essere distribuito
- **Pusher**: distribuisce il modello approvato a un endpoint di produzione

In [ ]:
# La instalación de tfx puede tardar varios minutos: instala Apache Beam,
# TensorFlow Data Validation, TensorFlow Transform y TensorFlow Model Analysis,
# entre otras dependencias. Ten paciencia la primera vez.
try:
    import colab
    IN_COLAB = True
    !pip install --upgrade pip -q
except ImportError:
    IN_COLAB = False

!pip install -U tfx -q


Se sei in Colab, è possibile che tu debba riavviare l'ambiente di esecuzione dopo l'installazione. Fallo dal menu **Runtime** > **Restart runtime** se vedi errori di import.

In [ ]:
import os
import shutil
import tempfile
import urllib.request

import tensorflow as tf
from tfx import v1 as tfx
from absl import logging

logging.set_verbosity(logging.INFO)

print('TensorFlow version:', tf.__version__)
print('TFX version:', tfx.__version__)


## Prepariamo i dati: Palmer Penguins

Useremo il dataset [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/), che contiene misurazioni di 344 pinguini di tre specie.

In [ ]:
PIPELINE_NAME = "penguin-simple"
PIPELINE_ROOT = os.path.join('pipelines', PIPELINE_NAME)
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

DATA_ROOT = tempfile.mkdtemp(prefix='tfx-data')
_data_url = ('https://raw.githubusercontent.com/tensorflow/tfx/master/tfx/'
             'examples/penguin/data/labelled/penguins_processed.csv')
_data_filepath = os.path.join(DATA_ROOT, "data.csv")
urllib.request.urlretrieve(_data_url, _data_filepath)

with open(_data_filepath) as f:
    for _ in range(5):
        print(f.readline().strip())


`species` vale 0, 1 o 2 (una delle tre specie) e il resto delle colonne sono già numeriche, così non abbiamo bisogno di molto preprocessing per questo esempio.

In [ ]:
_trainer_module_file = 'penguin_trainer.py'


In [ ]:
%%writefile {_trainer_module_file}

from typing import List
from absl import logging
import tensorflow as tf
from tensorflow import keras
from tensorflow_transform.tf_metadata import schema_utils

from tfx import v1 as tfx
from tfx_bsl.public import tfxio
from tensorflow_metadata.proto.v0 import schema_pb2

_FEATURE_KEYS = [
    'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'
]
_LABEL_KEY = 'species'

_TRAIN_BATCH_SIZE = 20
_EVAL_BATCH_SIZE = 10

# Como todavia no usamos un esquema generado por SchemaGen, describimos
# manualmente el formato esperado de cada feature.
_FEATURE_SPEC = {
    **{
        feature: tf.io.FixedLenFeature(shape=[1], dtype=tf.float32)
        for feature in _FEATURE_KEYS
    },
    _LABEL_KEY: tf.io.FixedLenFeature(shape=[1], dtype=tf.int64)
}


def _input_fn(file_pattern: List[str],
              data_accessor: tfx.components.DataAccessor,
              schema: schema_pb2.Schema,
              batch_size: int = 200) -> tf.data.Dataset:
  """Convierte los TFRecords que produce ExampleGen en un tf.data.Dataset."""
  return data_accessor.tf_dataset_factory(
      file_pattern,
      tfxio.TensorFlowDatasetOptions(
          batch_size=batch_size, label_key=_LABEL_KEY),
      schema=schema).repeat()


def _build_keras_model() -> tf.keras.Model:
  """Crea una red densa (DNN) sencilla para clasificar pinguinos."""
  inputs = [keras.layers.Input(shape=(1,), name=f) for f in _FEATURE_KEYS]
  d = keras.layers.concatenate(inputs)
  for _ in range(2):
    d = keras.layers.Dense(8, activation='relu')(d)
  outputs = keras.layers.Dense(3)(d)

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=keras.optimizers.Adam(1e-2),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy()])

  model.summary(print_fn=logging.info)
  return model


# TFX llama a esta funcion: es el punto de entrada del componente Trainer.
def run_fn(fn_args: tfx.components.FnArgs):
  schema = schema_utils.schema_from_feature_spec(_FEATURE_SPEC)

  train_dataset = _input_fn(
      fn_args.train_files, fn_args.data_accessor, schema,
      batch_size=_TRAIN_BATCH_SIZE)
  eval_dataset = _input_fn(
      fn_args.eval_files, fn_args.data_accessor, schema,
      batch_size=_EVAL_BATCH_SIZE)

  model = _build_keras_model()
  model.fit(
      train_dataset,
      steps_per_epoch=fn_args.train_steps,
      validation_data=eval_dataset,
      validation_steps=fn_args.eval_steps)

  # TFX espera encontrar el modelo guardado exactamente en esta ruta.
  model.save(fn_args.serving_model_dir, save_format='tf')


## Definiamo e eseguiamo il pipeline

Ora concateniamo i tre componenti: `ExampleGen`, `StatisticsGen` e `Trainer`.

In [ ]:
def _create_pipeline(pipeline_name: str, pipeline_root: str, data_root: str,
                      module_file: str, serving_model_dir: str,
                      metadata_path: str) -> tfx.dsl.Pipeline:
  """Pipeline de tres componentes: ExampleGen, Trainer, Pusher."""
  # 1. Ingiere el CSV y lo divide en entrenamiento/evaluacion.
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # 2. Entrena el modelo con el codigo de penguin_trainer.py.
  trainer = tfx.components.Trainer(
      module_file=module_file,
      examples=example_gen.outputs['examples'],
      train_args=tfx.proto.TrainArgs(num_steps=100),
      eval_args=tfx.proto.EvalArgs(num_steps=5))

  # 3. Publica el modelo entrenado en el sistema de archivos.
  pusher = tfx.components.Pusher(
      model=trainer.outputs['model'],
      push_destination=tfx.proto.PushDestination(
          filesystem=tfx.proto.PushDestination.Filesystem(
              base_directory=serving_model_dir)))

  components = [example_gen, trainer, pusher]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


In [ ]:
# LocalDagRunner ejecuta el pipeline directamente en esta maquina; es ideal
# para desarrollo. En produccion se usan orquestadores como Apache Airflow
# o Kubeflow Pipelines.
tfx.orchestration.LocalDagRunner().run(
    _create_pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=PIPELINE_ROOT,
        data_root=DATA_ROOT,
        module_file=_trainer_module_file,
        serving_model_dir=SERVING_MODEL_DIR,
        metadata_path=METADATA_PATH))


Se tutto è andato bene, dovresti vedere `INFO:absl:Component Pusher is finished.` alla fine. I componenti hanno registrato i loro output in una directory locale.

In [ ]:
!find {SERVING_MODEL_DIR}


## Cosa aggiunge TensorFlow Data Validation (TFDV)

Prima di fidarti di un modello, devi capire i tuoi dati. TFDV calcola statistiche dettagliate e rileva anomalie (ad es. un valore fuori dall'intervallo atteso).

In [ ]:
SCHEMA_PIPELINE_NAME = "penguin-schema"
SCHEMA_PIPELINE_ROOT = os.path.join('pipelines', SCHEMA_PIPELINE_NAME)
SCHEMA_METADATA_PATH = os.path.join('metadata', SCHEMA_PIPELINE_NAME, 'metadata.db')


def _create_schema_pipeline(pipeline_name, pipeline_root, data_root, metadata_path):
  """Pipeline corto solo para generar y revisar el esquema de los datos."""
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # NUEVO: calcula estadisticas descriptivas de cada feature.
  statistics_gen = tfx.components.StatisticsGen(
      examples=example_gen.outputs['examples'])

  # NUEVO: infiere un esquema (tipos, rangos) a partir de esas estadisticas.
  schema_gen = tfx.components.SchemaGen(
      statistics=statistics_gen.outputs['statistics'],
      infer_feature_shape=True)

  components = [example_gen, statistics_gen, schema_gen]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


tfx.orchestration.LocalDagRunner().run(
    _create_schema_pipeline(
        pipeline_name=SCHEMA_PIPELINE_NAME,
        pipeline_root=SCHEMA_PIPELINE_ROOT,
        data_root=DATA_ROOT,
        metadata_path=SCHEMA_METADATA_PATH))


In [ ]:
# Localizamos el esquema generado a traves del metadata del pipeline
# y lo copiamos a una carpeta fija, como harias antes de revisarlo a mano.
from tfx.orchestration.metadata import Metadata
from tfx.types import standard_component_specs
from tfx.orchestration.portable.mlmd import execution_lib


def get_latest_artifacts(metadata, pipeline_name, component_id):
  context = metadata.store.get_context_by_type_and_name(
      'node', f'{pipeline_name}.{component_id}')
  executions = metadata.store.get_executions_by_context(context.id)
  latest_execution = max(executions, key=lambda e: e.last_update_time_since_epoch)
  return execution_lib.get_output_artifacts(metadata, latest_execution.id)


metadata_connection_config = tfx.orchestration.metadata.sqlite_metadata_connection_config(
    SCHEMA_METADATA_PATH)

with Metadata(metadata_connection_config) as metadata_handler:
  schema_gen_output = get_latest_artifacts(
      metadata_handler, SCHEMA_PIPELINE_NAME, 'SchemaGen')
  schema_artifacts = schema_gen_output[standard_component_specs.SCHEMA_KEY]

SCHEMA_PATH = 'schema'
os.makedirs(SCHEMA_PATH, exist_ok=True)
_generated_path = os.path.join(schema_artifacts[0].uri, 'schema.pbtxt')
shutil.copy(_generated_path, SCHEMA_PATH)

print(open(os.path.join(SCHEMA_PATH, 'schema.pbtxt')).read())


Dovresti vedere quattro feature `FLOAT` e una `species` di tipo `INT`. Con questo schema, ExampleValidator può controllare se i nuovi dati rientrano nel formato atteso.

## Cosa aggiunge TensorFlow Transform (TFT)

I nostri feature venivano normalizzati per il notebook, ma in un pipeline di produzione il preprocessing deve essere ripetibile e coerente. TFT lo automatizza: applica la stessa trasformazione ai dati di allenamento, validazione e produzione.

In [ ]:
import pprint
import tensorflow_transform as tft
import tensorflow_transform.beam as tft_beam
from tensorflow_transform.tf_metadata import dataset_metadata, schema_utils

raw_data = [
    {'x': 1, 'y': 1, 's': 'hello'},
    {'x': 2, 'y': 2, 's': 'world'},
    {'x': 3, 'y': 3, 's': 'hello'},
]

raw_data_metadata = dataset_metadata.DatasetMetadata(
    schema_utils.schema_from_feature_spec({
        'y': tf.io.FixedLenFeature([], tf.float32),
        'x': tf.io.FixedLenFeature([], tf.float32),
        's': tf.io.FixedLenFeature([], tf.string),
    }))


def preprocessing_fn(inputs):
  """Funcion que TFT ejecutara igual en entrenamiento y en servicio."""
  x, y, s = inputs['x'], inputs['y'], inputs['s']
  # tft.mean() hace una pasada completa sobre TODO el dataset; no es una
  # operacion por ejemplo, a diferencia de una resta normal de TensorFlow.
  x_centered = x - tft.mean(x)
  y_normalized = tft.scale_to_0_1(y)
  # Convierte cadenas de texto en indices segun su frecuencia.
  s_integerized = tft.compute_and_apply_vocabulary(s)
  return {
      'x_centered': x_centered,
      'y_normalized': y_normalized,
      's_integerized': s_integerized,
  }


with tft_beam.Context(temp_dir=tempfile.mkdtemp()):
  transformed_dataset, _ = (
      (raw_data, raw_data_metadata)
      | tft_beam.AnalyzeAndTransformDataset(preprocessing_fn))

transformed_data, _ = transformed_dataset
print('Datos originales:')
pprint.pprint(raw_data)
print('Datos transformados:')
pprint.pprint(transformed_data)


Nota che `x_centered` rimane centrato intorno a 0 usando la media di **tutti i dati di allenamento**: non è calcolato per ogni esempio singolo (quello sarebbe leakage).

## Cosa aggiunge TensorFlow Model Analysis (TFMA)

Un modello può avere una precisione globale buona ma funzionare male su sottogruppi specifici (ad es. una specie di pinguino). TFMA sliced la valutazione per esplorare le prestazioni per sottogruppo.

## Metadata ed elaborazione distribuita (concetto)

Due pezzi in più completano il sistema: TFDV, TFT e TFMA usano lo schema TFX (metadata che descrive i dati) e possono essere eseguiti su infrastrutture distribuite (Google Cloud Dataflow, Apache Beam) per dataset enormi.

## Riepilogo

- TFX porta un modello da un notebook a un sistema di produzione affidabile
- Automatizza caricamento dei dati, preprocessing, addestramento, valutazione e distribuzione
- I componenti usano metadati e versionamento per tracciare cosa è successo
- Puoi estendere un pipeline TFX aggiungendo i tuoi componenti personalizzati